# Computer Vision — Session 2 Hands-on

## Building an Image Data Pipeline

**AI Builders Bootcamp**

---

In Session 1 we learned that an image is just **numbers**: a grid of pixels stored in a NumPy array.

In this notebook we take the next step. A single image is not enough to train a model — we need a
**pipeline** that can take *any* image from a folder and turn it into something a neural network can eat.

We will build that pipeline on a real dataset: the **Dog and Cat Classification Dataset** from
Kaggle — around 25,000 photographs of cats and dogs, downloaded with `kagglehub`. Real photos means
real problems: inconsistent sizes, corrupted files, and messy backgrounds that no simple threshold
can handle. That is the point.

```text
Image
 ↓
Preprocessing
 ↓
Augmentation
 ↓
Tensor
 ↓
Dataset
 ↓
DataLoader
```

> **Before training a model, we need a reliable and reusable data pipeline.**

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Apply common image preprocessing techniques
- Visualize the effect of image transformations
- Understand image augmentation
- Split image data into train, validation, and test sets
- Understand how data leakage can happen
- Inspect class distribution
- Convert images into PyTorch tensors
- Build a custom PyTorch `Dataset`
- Use a `DataLoader`
- Work with batches of images

---

## What we will *not* do today

- We will **not** train a neural network
- We will **not** build a CNN
- We will **not** use a pretrained model

Today is about the **data**. The model comes next session.

---

# 2. Setup

If you are missing any library, run this once in your terminal (with your virtual environment active):

```bash
pip install numpy matplotlib opencv-python torch torchvision kagglehub
```

What each library does in this notebook:

| Library | Role |
|---|---|
| **OpenCV** (`cv2`) | image processing: load, resize, blur, threshold, edges, contours, morphology |
| **NumPy** | numerical operations on the pixel array |
| **Matplotlib** | visualization |
| **PyTorch** (`torch`) | tensors — the data structure deep learning models use |
| **`Dataset`** | defines *how to access one training example* |
| **`DataLoader`** | loads many examples at once, in batches |
| **`torchvision.transforms`** | ready-made preprocessing and augmentation steps |
| **kagglehub** | downloads our dataset from Kaggle |

In [ ]:
import os
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import kagglehub

print("OpenCV     :", cv2.__version__)
print("NumPy      :", np.__version__)
print("PyTorch    :", torch.__version__)
print("kagglehub  :", kagglehub.__version__)

### Reproducibility

Machine learning uses randomness in many places (shuffling, augmentation, splitting).
If we fix the **seed**, we get the same "random" result every time we run the notebook.
That makes our experiments repeatable — and makes debugging much easier.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Random seed set to", SEED)

---

# 3. Where Is Our Data?

Today we use the **Dog and Cat Classification Dataset** from Kaggle: roughly 25,000 real
photographs, split into two classes.

A very common way to store an image classification dataset is **one folder per class**, and this
dataset follows exactly that convention:

```text
PetImages/
│
├── Cat/
│   ├── 0.jpg
│   ├── 1.jpg
│   └── ...
│
└── Dog/
    ├── 0.jpg
    ├── 1.jpg
    └── ...
```

Three ideas to remember:

1. **Folder names represent labels.** The folder `Cat` *is* the label of every image inside it.
2. **Each image belongs to exactly one class.** No image lives in two folders.
3. **The pipeline's job** is to convert `(image file, folder name)` into `(tensor, integer label)` —
   because a model cannot read a `.jpg` and cannot read the word `"Cat"`.

Nothing below is hard-coded to cats and dogs. The class names are read from the folder names, so the
same code works for **any** number of classes — see section 3.4 to point it at your own data.

## 3.1 Download the dataset with kagglehub

`kagglehub` downloads a dataset and returns the local path where it landed. It **caches** the
download, so the first run takes a few minutes and every run after that is instant.

### One-time setup: Kaggle credentials

Kaggle requires an account before it will hand over a dataset. If you have never done this:

1. Create a free account at [kaggle.com](https://www.kaggle.com)
2. Go to **Settings → API → Create New Token** — this downloads a `kaggle.json` file
3. Put that file at `~/.kaggle/kaggle.json` (on Windows: `C:\Users\<you>\.kaggle\kaggle.json`)

Alternatively, run `kagglehub.login()` in a cell and paste your username and key when prompted.

If you skip this step the next cell will fail with an authentication error — that is expected, not a
bug in your code.

In [ ]:
DATASET_SLUG = "bhavikjikadara/dog-and-cat-classification-dataset"

download_path = kagglehub.dataset_download(DATASET_SLUG)

print("Dataset downloaded to:")
print(" ", download_path)

## 3.2 Find the folder that holds the class folders

`kagglehub` gives us the root of the download, but datasets are often wrapped in one or two extra
folders (`versions/1/PetImages/Cat`, and so on). Rather than hard-coding a path that breaks the
moment the dataset is repackaged, we search downwards for the first folder whose subfolders
actually contain images.

This is a small piece of defensive code, and it is the reason this notebook keeps working when you
swap in a different dataset.

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")


def find_class_root(start_dir):
    """Walk down from start_dir and return the first folder whose subfolders contain images."""
    for current_dir, sub_dirs, _ in os.walk(start_dir):
        candidates = sorted(d for d in sub_dirs if not d.startswith("."))
        if not candidates:
            continue

        every_subfolder_has_images = all(
            any(name.lower().endswith(IMAGE_EXTENSIONS) for name in os.listdir(os.path.join(current_dir, d)))
            for d in candidates
        )
        if every_subfolder_has_images:
            return current_dir

    raise FileNotFoundError(f"Could not find any class folders under {start_dir}")


DATA_DIR = find_class_root(download_path)

print("Class folders live in:")
print(" ", DATA_DIR)
print()
print("Contents:", sorted(os.listdir(DATA_DIR)))

## 3.3 Read the class names

The folder names *are* our labels. We sort them so the mapping is stable — if the order changed
between runs, `Cat` could be label `0` today and label `1` tomorrow, which would silently corrupt
every experiment you run.

In [ ]:
class_names = sorted(
    name for name in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, name))
)

# Models work with integers, not words, so we map each class name to a number.
class_to_index = {class_name: index for index, class_name in enumerate(class_names)}

print("Classes found :", class_names)
print("Label mapping :", class_to_index)

## 3.4 Using your own dataset instead

Everything from here on only depends on `DATA_DIR` pointing at a folder-per-class layout. To use
your own images, skip the two cells above and set it directly:

```python
DATA_DIR = "dataset"   # a folder containing one subfolder per class
```

```text
your_project/
├── 02_image_processing_data_pipeline.ipynb
└── dataset/
    ├── class_1/
    └── class_2/
```

## 3.5 Collecting image paths and labels

We never load the whole dataset into memory — 25,000 photographs would not fit comfortably, and we
do not need them all at once anyway. Instead we build **two parallel lists**:

```text
image_paths[i]  ->  ".../PetImages/Cat/0.jpg"
labels[i]       ->  0
```

`image_paths[i]` and `labels[i]` always describe the *same* example. This pairing is the
backbone of the whole pipeline.

### Working with a subset

25,000 images make every cell in this notebook slow, and we are not training anything today. So we
take a **random sample** of `MAX_IMAGES_PER_CLASS` images per class.

Two details that matter:

- We sample **randomly**, not "the first 500". File order is often meaningful (sorted by date, by
  source, by subject), so taking a prefix can quietly hand you a biased subset.
- We sample the **same** images every run, because the random generator is seeded.

Set `MAX_IMAGES_PER_CLASS = None` to use the full dataset.

### Real datasets contain broken files

This dataset ships with **corrupted files** — images that are zero bytes, truncated, or not actually
images at all. They will happily sit in your list of paths and then crash your training loop several
minutes in, which is a miserable way to find out.

We deal with them in two stages, because the two checks cost wildly different amounts:

| Check | Cost | Applied to |
|---|---|---|
| Is the file zero bytes? | a `stat` call — microseconds | **every** file in the dataset |
| Can OpenCV actually decode it? | full decode — milliseconds | only the images we sampled |

The cheap check runs over all 25,000 files, so a zero-byte file is caught whether or not our random
sample happened to pick it. The expensive check then runs over the few hundred we kept.

In [ ]:
MAX_IMAGES_PER_CLASS = 500   # set to None to use every image


def collect_image_paths(data_dir, class_names, class_to_index, max_per_class=None):
    """Build parallel lists of image paths and integer labels, optionally sampling each class."""
    rng = np.random.default_rng(SEED)
    image_paths = []
    labels = []
    empty_files = []

    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        file_names = sorted(
            name for name in os.listdir(class_dir)
            if name.lower().endswith(IMAGE_EXTENSIONS)
        )

        # A zero-byte file is never a valid image, and checking the size is cheap enough
        # to do for every file in the dataset rather than only the ones we sample.
        usable_paths = []
        for file_name in file_names:
            path = os.path.join(class_dir, file_name)
            if os.path.getsize(path) == 0:
                empty_files.append(path)
            else:
                usable_paths.append(path)

        if max_per_class is not None and len(usable_paths) > max_per_class:
            chosen = rng.choice(len(usable_paths), size=max_per_class, replace=False)
            usable_paths = [usable_paths[i] for i in sorted(chosen)]

        image_paths.extend(usable_paths)
        labels.extend([class_to_index[class_name]] * len(usable_paths))

    return image_paths, labels, empty_files


image_paths, labels, empty_files = collect_image_paths(
    DATA_DIR, class_names, class_to_index, max_per_class=MAX_IMAGES_PER_CLASS
)

print("Images collected  :", len(image_paths))
print("Zero-byte files   :", len(empty_files))
for path in empty_files[:5]:
    print("   skipped:", path)
print()
print("First example path :", image_paths[0])
print("First example label:", labels[0], "->", class_names[labels[0]])

## 3.6 Verify that every image actually decodes

Zero bytes is the obvious failure. The nastier one is a file that has a perfectly reasonable size and
still cannot be decoded — truncated downloads, HTML error pages saved with a `.jpg` extension, images
in a format OpenCV was not built to read.

`cv2.imread` returns `None` for all of these **instead of raising an error**, so they slip through
silently. The only reliable test is to try:

```text
Every path we kept
        ↓
   Try to load it
        ↓
 Keep the ones that work
```

The check costs a few seconds and saves you an afternoon.

In [ ]:
def keep_readable_images(image_paths, labels):
    """Drop any file that OpenCV cannot actually decode."""
    good_paths = []
    good_labels = []
    broken_paths = []

    for path, label in zip(image_paths, labels):
        if cv2.imread(path) is None:
            broken_paths.append(path)
        else:
            good_paths.append(path)
            good_labels.append(label)

    return good_paths, good_labels, broken_paths


image_paths, labels, broken_paths = keep_readable_images(image_paths, labels)

print("Usable images :", len(image_paths))
print("Broken images :", len(broken_paths))
for path in broken_paths[:5]:
    print("   dropped:", path)

---

# 4. Load and Inspect One Image

Back to Session 1 for a moment. When OpenCV loads an image we get a **NumPy array**.

One OpenCV detail that trips up everyone at least once:

```text
OpenCV loads images as   B G R
Matplotlib expects       R G B
```

If your image looks blue-ish, you forgot the conversion.

In [ ]:
# Change this to any image path you like, or leave it as None to use the first dataset image.
SAMPLE_IMAGE_PATH = None

sample_path = SAMPLE_IMAGE_PATH if SAMPLE_IMAGE_PATH else image_paths[0]

image_bgr = cv2.imread(sample_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

height, width, channels = image_rgb.shape

print("Path      :", sample_path)
print("Type      :", type(image_rgb))
print("Data type :", image_rgb.dtype)
print("Shape     :", image_rgb.shape, " (height, width, channels)")
print()
print("Height    :", height)
print("Width     :", width)
print("Channels  :", channels)
print()
print("Pixel value range:", image_rgb.min(), "to", image_rgb.max())

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(image_rgb)
plt.title(f"Original image {image_rgb.shape}")
plt.axis("off")
plt.show()

### A small helper for the rest of the notebook

We are going to compare "before and after" images many times, so let's write one small
plotting function instead of repeating the same Matplotlib code everywhere.

In [ ]:
def show_images(images, titles, cmap=None, columns=None):
    """Display a row (or grid) of images side by side."""
    columns = columns or len(images)
    rows = int(np.ceil(len(images) / columns))

    plt.figure(figsize=(4 * columns, 4 * rows))
    for i, (image, title) in enumerate(zip(images, titles)):
        plt.subplot(rows, columns, i + 1)
        plt.imshow(image, cmap=cmap)
        plt.title(title, fontsize=11)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

---

# 5. Preparing Images

> Before images enter a model, we often need to transform them into a consistent format.

Photos come in every shape and size. Neural networks are much less flexible: they expect
every input to have **exactly the same shape** and a **predictable value range**.

Preprocessing is the set of steps that makes that true.

## 5.1 Resizing

```text
Images of many sizes
        ↓
      Resize
        ↓
One consistent size
```

Why consistent dimensions matter:

- A batch is a single tensor — you cannot stack a 400×300 image and a 1024×768 image into one block of numbers.
- The model's layers are built for a fixed input size.
- Smaller images mean less memory and faster training.

`224 × 224` is a very common choice, so we will use it throughout.

First, let's confirm the problem is real — here are the shapes of the first few images in our dataset:

In [ ]:
for path in image_paths[:6]:
    shape = cv2.imread(path).shape
    print(f"{shape}   {os.path.basename(path)}")

Every one of them is different. Without resizing, we could not stack even two of these into a single
tensor.

In [ ]:
IMAGE_SIZE = 224

resized_image = cv2.resize(image_rgb, (IMAGE_SIZE, IMAGE_SIZE))

print("Original shape :", image_rgb.shape)
print("Resized shape  :", resized_image.shape)

show_images(
    [image_rgb, resized_image],
    [f"Original {image_rgb.shape[:2]}", f"Resized {resized_image.shape[:2]}"],
)

**Note on aspect ratio:** `cv2.resize` stretches the image to fit the target size. If the original was
not square, the result looks slightly squashed. That is usually acceptable, but it is a real choice —
cropping is the alternative.

**Challenge:** resize the image to `64 × 64` and then back to `224 × 224`. What happened to the detail?
Why can resizing never *add* information?

## 5.2 Normalization

Pixels are stored as integers from `0` to `255`. Neural networks train much more comfortably with
small floating point numbers centred around zero.

There are **two separate steps**, and beginners often mix them up:

```text
Step 1 — Scaling                 Step 2 — Normalization
0 – 255  →  0.0 – 1.0            (x - mean) / std
```

Let's do step 1 by hand with NumPy so there is no magic.

In [ ]:
scaled_image = image_rgb.astype(np.float32) / 255.0

print("Before scaling")
print("  dtype :", image_rgb.dtype)
print("  range : [{}, {}]".format(image_rgb.min(), image_rgb.max()))
print()
print("After scaling")
print("  dtype :", scaled_image.dtype)
print("  range : [{:.3f}, {:.3f}]".format(scaled_image.min(), scaled_image.max()))
print()
print("Top-left pixel before:", image_rgb[0, 0])
print("Top-left pixel after :", scaled_image[0, 0])

### Step 2 — normalization with mean and standard deviation

Scaling puts the values in `[0, 1]`. Normalization then shifts and stretches them:

```text
normalized = (value - mean) / std
```

With `mean = 0.5` and `std = 0.5`, a value of `0.0` becomes `-1.0` and a value of `1.0` becomes `1.0`,
so the data ends up centred around zero in `[-1, 1]`. That is a good, simple default for beginners.

You will often see other numbers, for example `mean = [0.485, 0.456, 0.406]`. Those are the channel
statistics of the ImageNet dataset, used when a model was *trained* on ImageNet.

> **The exact preprocessing depends on the model you plan to use.**
> Use whatever the model was trained with. We will revisit this in the CNN session.

In [ ]:
normalized_image = (scaled_image - 0.5) / 0.5

print("Scaled range     : [{:.2f}, {:.2f}]".format(scaled_image.min(), scaled_image.max()))
print("Normalized range : [{:.2f}, {:.2f}]".format(normalized_image.min(), normalized_image.max()))

### The same thing with torchvision

We did that by hand to see what happens. In practice `torchvision.transforms` does both steps for us:

- `transforms.ToTensor()` → converts to a tensor, scales `0–255` to `0.0–1.0`, and reorders
  the axes from `H × W × C` to `C × H × W`
- `transforms.Normalize(mean, std)` → applies `(value - mean) / std` per channel

Watch the **shape** change in the output below — that reordering is important and we will come back to it.

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

tensor_image = to_tensor(image_rgb)
normalized_tensor = normalize(tensor_image)

print("NumPy image     :", image_rgb.shape, image_rgb.dtype, "  (H, W, C)")
print("After ToTensor  :", tuple(tensor_image.shape), tensor_image.dtype, "  (C, H, W)")
print()
print("ToTensor  range : [{:.2f}, {:.2f}]".format(tensor_image.min(), tensor_image.max()))
print("Normalize range : [{:.2f}, {:.2f}]".format(normalized_tensor.min(), normalized_tensor.max()))

---

# 6. Classical Image Processing

Before deep learning, computer vision was built almost entirely from operations like the ones in this
section. They are still extremely useful today — for cleaning up images, for extracting simple features,
and for understanding what is actually in your data.

The goal here is **exploration**, not building a complicated pipeline. Run each cell, look at the
picture, and build intuition.

## 6.1 Thresholding

Thresholding turns a grayscale image into a **binary** (black and white) image:

```text
Pixel
 ↓
Compare with threshold
 ↓
Black or White
```

Every pixel brighter than the threshold becomes white (`255`); everything else becomes black (`0`).
It is the simplest possible way to separate a **foreground** object from the **background**.

Let's start with the obvious choice — the middle of the `0–255` range:

In [ ]:
gray_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

THRESHOLD_VALUE = 127
_, binary_fixed = cv2.threshold(gray_image, THRESHOLD_VALUE, 255, cv2.THRESH_BINARY)

white_share = 100 * np.count_nonzero(binary_fixed) / binary_fixed.size

print("Grayscale shape :", gray_image.shape, " (no colour channel anymore)")
print("Binary values   :", np.unique(binary_fixed))
print(f"White pixels    : {white_share:.1f}%")

show_images(
    [gray_image, binary_fixed],
    ["Grayscale", f"Fixed threshold (> {THRESHOLD_VALUE})"],
    cmap="gray",
)

### The problem with a fixed threshold

Look at the percentage of white pixels above. On a dark photo `127` swallows almost everything into
black; on a bright photo almost everything turns white. **There is no single number that works for
every image** — and a real dataset has both kinds of photo in it.

Since we want the rest of this section to work on *whatever* image you happen to have, we let OpenCV
pick the threshold from the image itself. Adding the `cv2.THRESH_OTSU` flag tells it to look at the
histogram of pixel values and choose the split that separates dark from light best.

You do not need to know how it works. What matters is the idea:

```text
Fixed threshold  → you guess a number
Otsu threshold   → the image tells you the number
```

In [ ]:
otsu_value, binary_image = cv2.threshold(
    gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print(f"Fixed threshold : {THRESHOLD_VALUE}")
print(f"Otsu threshold  : {otsu_value:.0f}   <- chosen from this image")

show_images(
    [binary_fixed, binary_image],
    [f"Fixed (> {THRESHOLD_VALUE})", f"Otsu (> {otsu_value:.0f})"],
    cmap="gray",
)

We will use `binary_image` (the Otsu version) for the rest of this section.

**Challenge:** change `THRESHOLD_VALUE` to `60` and then to `200` and re-run the fixed-threshold cell.
What gets lost each time? Then try a few different photos from the dataset by changing
`SAMPLE_IMAGE_PATH` — how much does the best threshold move between images?

## 6.2 Blurring

Blurring replaces each pixel with a weighted average of its neighbours.

Why would we deliberately make an image *less* sharp?

- **Noise reduction** — random speckles get averaged away, real structures survive
- **Smoothing** — small irrelevant details disappear so the big shapes stand out
- It is a common **preparation step** before thresholding or edge detection

The kernel size (`(9, 9)` below) controls how strong the blur is. It must be an odd number.

In [ ]:
blurred_image = cv2.GaussianBlur(image_rgb, (9, 9), sigmaX=0)

show_images(
    [image_rgb, blurred_image],
    ["Original", "Gaussian blur (9 x 9)"],
)

**Challenge:** try kernel sizes `(3, 3)`, `(15, 15)` and `(31, 31)`. Where does "helpful smoothing"
turn into "I destroyed my image"?

## 6.3 Edge Detection

An **edge** is a place where pixel intensity changes sharply — the boundary between an object and
its background, for example.

Canny edge detection finds those locations and returns a binary image where white pixels are edges.
It takes two thresholds that control how strong a change must be to count as an edge. We will treat
the algorithm itself as a black box for now.

In [ ]:
edges = cv2.Canny(gray_image, threshold1=100, threshold2=200)

show_images(
    [gray_image, edges],
    ["Grayscale", "Canny edges"],
    cmap="gray",
)

Notice that edge detection throws away almost everything — colour, texture, brightness — and keeps
only *where things change*. That is a huge reduction of information, and sometimes exactly what you want.

**Challenge:** blur the image first, then run Canny on the blurred version. Do you get fewer noisy edges?

## 6.4 Contours

A **contour** is a curve joining all the continuous points along a boundary. Where edges are just
loose pixels, contours are complete **outlines of shapes** that you can count, measure and draw.

The pipeline is always the same:

```text
Image
 ↓
Grayscale
 ↓
Threshold or Edges
 ↓
Contours
```

We will take the **edges** branch. On a photograph a threshold usually produces one enormous white
region that runs off all four sides of the frame — so `findContours` hands back a single rectangle
the size of the image, which tells us nothing. An edge map does not have that problem: it traces
structures wherever they are.

Two practical steps come with it:

- **Blur first**, so we trace real structures instead of sensor noise
- **Ignore the short ones**, because a photograph produces hundreds of tiny fragments. Filtering by
  size is the simplest and most common way to separate "a real structure" from "a few stray pixels".

We measure a contour's size with `cv2.arcLength` — how long the traced curve is, in pixels. (There is
also `cv2.contourArea`, but that is only meaningful for contours that enclose a filled region; a
traced edge is a thin curve, so its area is close to zero.)

In [ ]:
blurred_gray = cv2.GaussianBlur(gray_image, (5, 5), sigmaX=0)
contour_edges = cv2.Canny(blurred_gray, threshold1=100, threshold2=200)

contours, _ = cv2.findContours(contour_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

MIN_CONTOUR_LENGTH = 100   # pixels

long_contours = [c for c in contours if cv2.arcLength(c, False) >= MIN_CONTOUR_LENGTH]

# Draw on a copy so the original image stays untouched.
contour_image = image_rgb.copy()
cv2.drawContours(contour_image, long_contours, -1, (0, 255, 0), thickness=2)

print("Contours found          :", len(contours))
print(f"Longer than {MIN_CONTOUR_LENGTH} pixels :", len(long_contours))
print()
for i, contour in enumerate(sorted(long_contours, key=lambda c: cv2.arcLength(c, False), reverse=True)[:5]):
    print(f"  contour {i}: {len(contour):>4} points, length = {cv2.arcLength(contour, False):>7.0f} px")

show_images(
    [image_rgb, cv2.cvtColor(contour_edges, cv2.COLOR_GRAY2RGB), contour_image],
    ["Original", "Edges", f"Contours ({len(long_contours)} kept)"],
)

`cv2.RETR_EXTERNAL` keeps only the **outermost** contours, which is usually what you want when you
are counting objects.

Notice how many contours the length filter threw away, and — more importantly — that not one of the
survivors is "the outline of the pet". On a photograph of a real animal, classical methods will never
hand you a clean silhouette: the background, the fur and the lighting all get in the way.

This is exactly the limitation that pushed computer vision towards **learned** features, and it is
worth seeing with your own eyes before the CNN session. Contours are wonderful for scanned documents,
manufactured parts and microscope slides. They are not going to tell a cat from a dog.

**Challenge:** print the longest contour with `max(contours, key=lambda c: cv2.arcLength(c, False))`.
Could you use its length as a simple feature to tell cats from dogs — without any machine learning?
Why not?

---

## 6.5 Morphological Operations

Morphological operations are image-processing techniques used mainly on **binary or grayscale
images** to modify and analyse the **shape and structure** of objects.

```text
Binary Image
     ↓
Morphological Operation
     ↓
Modified Image
```

They do not care about colour or texture. They only ask: *where is the foreground, and what shape is it?*

### The kernel (structuring element)

Every morphological operation slides a small window over the image:

```python
kernel = np.ones((5, 5), np.uint8)
```

The kernel defines the **neighbourhood** that is looked at around each pixel. A bigger kernel means
a stronger effect. That is all you need to know for now — we will build intuition by looking at pictures
rather than by studying the mathematics.

### Our test image

We already have a perfect test case: the binary image from section 6.1. Thresholding a real
photograph never produces clean shapes — it produces regions riddled with **small white specks** in
the background and **small black holes** inside the objects. Those are precisely the defects
morphological operations are designed to fix.

In [ ]:
kernel = np.ones((5, 5), np.uint8)


def foreground_percent(binary):
    """Share of white (foreground) pixels — a simple number to watch as we shrink and grow regions."""
    return 100 * np.count_nonzero(binary) / binary.size


print("Binary image shape :", binary_image.shape)
print("Kernel shape       :", kernel.shape)
print(f"Foreground         : {foreground_percent(binary_image):.1f}% of all pixels")

show_images([gray_image, binary_image], ["Grayscale", "Binary (our test case)"], cmap="gray")

### 6.5.1 Erosion

> **Erosion shrinks the foreground regions in an image.**

A white pixel stays white only if *all* its neighbours (inside the kernel) are also white.
Anything thin or isolated disappears.

In [ ]:
erosion = cv2.erode(binary_image, kernel, iterations=1)

print(f"Foreground before : {foreground_percent(binary_image):.1f}%")
print(f"Foreground after  : {foreground_percent(erosion):.1f}%")

show_images(
    [binary_image, erosion],
    ["Original binary image", "Erosion"],
    cmap="gray",
)

Erosion can:

- remove small white noise
- shrink objects
- separate objects that are slightly connected

**Observe:** *What happens to small objects after erosion?*
Look at the background specks — and notice that the main object got smaller too. That is the price of erosion.

### 6.5.2 Dilation

> **Dilation expands the foreground regions in an image.**

A pixel becomes white if *any* of its neighbours is white. It is the mirror image of erosion.

In [ ]:
dilation = cv2.dilate(binary_image, kernel, iterations=1)

print(f"Foreground before : {foreground_percent(binary_image):.1f}%")
print(f"Foreground after  : {foreground_percent(dilation):.1f}%")

show_images(
    [binary_image, dilation],
    ["Original binary image", "Dilation"],
    cmap="gray",
)

Dilation can:

- expand objects
- fill small gaps
- connect nearby regions

**Observe:** *What happens when two objects are very close to each other?*
They grow towards each other and can merge into one blob. Sometimes that is what you want
(joining broken parts of a letter); sometimes it destroys the very thing you were trying to count.

### 6.5.3 Opening

Opening is a **combination** of the two operations we just saw:

```text
Opening
   =
Erosion
   ↓
Dilation
```

Opening first shrinks the foreground and then expands it again.

```text
Small Noise
    ↓
Erosion removes it
    ↓
Dilation restores the main objects
```

The small specks are destroyed by the erosion and there is nothing left for the dilation to bring back.
The big object survives the erosion and is then restored to roughly its original size.

In [ ]:
opening = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel)

print(f"Original : {foreground_percent(binary_image):.1f}%")
print(f"Erosion  : {foreground_percent(erosion):.1f}%   <- shrank a lot")
print(f"Opening  : {foreground_percent(opening):.1f}%   <- noise gone, size mostly restored")

show_images(
    [binary_image, opening],
    ["Original binary image", "Opening (erode then dilate)"],
    cmap="gray",
)

Opening is commonly useful for:

- removing small foreground noise
- removing small objects
- smoothing object boundaries

Compare this with plain erosion: the background specks are gone in **both**, but with Opening the
main object kept its size.

### 6.5.4 Closing

Closing is the **opposite sequence**:

```text
Closing
   =
Dilation
   ↓
Erosion
```

```text
Small Gap / Hole
      ↓
Dilation expands the region
      ↓
Erosion restores the overall shape
```

The dilation swallows the small holes inside the object, and the erosion afterwards brings the outer
boundary back to where it started.

In [ ]:
closing = cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel)

print(f"Original : {foreground_percent(binary_image):.1f}%")
print(f"Dilation : {foreground_percent(dilation):.1f}%   <- grew a lot")
print(f"Closing  : {foreground_percent(closing):.1f}%   <- holes filled, size mostly restored")

show_images(
    [binary_image, closing],
    ["Original binary image", "Closing (dilate then erode)"],
    cmap="gray",
)

Closing can help:

- fill small holes inside objects
- close small gaps
- connect nearby regions

**Notice** that Closing did *not* remove the background specks — it made them slightly rounder.
Opening and Closing solve different problems. A very common recipe is to apply **Opening then Closing**:
first remove the noise, then fill the holes.

### 6.5.5 Morphological Gradient

The morphological gradient highlights the **boundaries** of objects.

```text
Morphological Gradient
        =
Dilation − Erosion
```

The intuition:

- **Dilation** expands the object
- **Erosion** shrinks the object
- The **difference** between them is exactly the ring of pixels around the boundary

The wider the kernel, the thicker the outline.

In [ ]:
gradient = cv2.morphologyEx(binary_image, cv2.MORPH_GRADIENT, kernel)

show_images(
    [binary_image, gradient],
    ["Original binary image", "Morphological gradient"],
    cmap="gray",
)

> Unlike Canny edge detection, which detects **intensity changes**, the morphological gradient finds
> boundaries based on the morphological **expansion and shrinking of image regions**.

Both give you outlines, but they arrive there in completely different ways — and the morphological
gradient works on the *shape* of regions you have already segmented.

### 6.5.6 Compare All Morphological Operations

Let's put everything side by side.

```text
Original
   │
   ├── Erosion  → Shrinks objects
   │
   ├── Dilation → Expands objects
   │
   ├── Opening  → Removes small foreground noise
   │
   ├── Closing  → Fills small gaps and holes
   │
   └── Gradient → Highlights object boundaries
```

In [ ]:
show_images(
    [binary_image, erosion, dilation, opening, closing, gradient],
    ["Original", "Erosion", "Dilation", "Opening", "Closing", "Gradient"],
    cmap="gray",
    columns=3,
)

| Operation | Main Effect |
|---|---|
| Erosion | Shrinks foreground regions |
| Dilation | Expands foreground regions |
| Opening | Removes small foreground noise |
| Closing | Fills small gaps and holes |
| Morphological Gradient | Highlights object boundaries |

**Challenge:** change the kernel to `np.ones((11, 11), np.uint8)` and re-run the comparison.
Which operations become destructive?

> **Morphological operations are useful when the shape and structure of objects are important.**

---

# 7. Making Training Data More Diverse

> **Augmentation creates modified versions of training images.**

A model that has only ever seen a cat facing left may fail on a cat facing right. Augmentation shows
the model flipped, rotated and slightly recoloured versions of the same photo, so it learns
*"cat"* instead of *"cat facing left, top-left corner, bright lighting"*.

```text
1 training image
       ↓
   Augmentation
       ↓
Many different-looking versions of the same example
```

Two rules:

1. Augmentation is applied **on the fly**, every time an image is loaded — so the model sees a
   slightly different version each epoch. We do not save new files to disk.
2. Augmentation is for **training data only**. Validation and test data must stay fixed, otherwise
   your scores change every time you evaluate.

The most important constraint:

> **Every augmented image must still belong to the same class.**

In [ ]:
augmentation_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

print(augmentation_transform)

Reading the pipeline top to bottom:

| Step | What it does |
|---|---|
| `ToPILImage()` | our image is a NumPy array; torchvision transforms expect a PIL image |
| `RandomHorizontalFlip()` | mirrors the image left-right, 50% of the time |
| `RandomRotation(15)` | rotates by a random angle between −15° and +15° |
| `ColorJitter(...)` | randomly changes brightness and contrast by up to 20% |
| `ToTensor()` | converts to a `C × H × W` tensor with values in `[0, 1]` |

Notice that the transforms are **random**: applying the same pipeline twice to the same image gives
two different results. That is exactly the point.

In [ ]:
def tensor_to_image(tensor):
    """Convert a C x H x W tensor in [0, 1] back to an H x W x C array we can display."""
    return tensor.permute(1, 2, 0).numpy()


augmented_versions = [tensor_to_image(augmentation_transform(image_rgb)) for _ in range(5)]

show_images(
    [image_rgb] + augmented_versions,
    ["Original"] + [f"Augmented #{i + 1}" for i in range(5)],
    columns=3,
)

### Discussion — would flipping or rotating always make sense?

Augmentation is not free. Each transform makes an assumption about your data, and if the assumption
is wrong you are teaching the model something false.

**For cats and dogs specifically:**

- **Horizontal flip — safe.** A mirrored cat is still a cat. Photographers point the camera from
  either side, so both versions genuinely occur in the real world.
- **Small rotation — safe.** Hand-held photos are rarely perfectly level, so ±15° is realistic.
- **Vertical flip — a bad idea.** People do not photograph upside-down animals. You would spend model
  capacity learning a situation the model will never meet at test time.
- **Heavy colour jitter — risky.** Coat colour is a real signal. Push saturation and hue far enough
  and you turn a ginger cat into an animal that does not exist.

That is why our pipeline uses horizontal flip, mild rotation and *gentle* colour jitter — and no
vertical flip.

Now the general principle, with examples from other domains:

**Example 1 — handwritten digits.** A horizontal flip turns a `2` into a mirrored shape that is not a
`2` at all. A vertical flip turns `6` into something close to `9`. You would be handing the model
images with the *wrong label*.

**Example 2 — medical or road imagery.** In a chest X-ray, whether the heart appears on the left or
the right is diagnostically meaningful; flipping destroys that. A rotated road sign in a
self-driving dataset may never occur in reality, so you spend model capacity learning a situation
that never happens.

**Example 3 — too much colour jitter.** If you are classifying ripe vs unripe fruit, aggressive
brightness and hue changes can literally turn one class into the other.

The rule of thumb:

> Only use an augmentation if the transformed image is something you could plausibly encounter
> in the real world **with the same label**.

**Challenge:** for *your* dataset, write down two augmentations that make sense and one that would be harmful.

---

# 8. Dataset Inspection

Before touching a model, look at your data. Almost every "the model doesn't work" story starts with
a dataset problem that would have been visible in five minutes of inspection: wrong labels, corrupted
files, a class with only three examples, images that are all identical.

We will check three things:

1. How many images per class?
2. What do the images actually look like?
3. Is the dataset balanced?

In [ ]:
label_array = np.array(labels)

print("Classes     :", class_names)
print("Total images:", len(image_paths))
print()
for class_name in class_names:
    count = int(np.sum(label_array == class_to_index[class_name]))
    percentage = 100 * count / len(labels)
    print(f"  {class_name:<12} {count:>5} images  ({percentage:5.1f}%)")

In [ ]:
counts = [int(np.sum(label_array == index)) for index in range(len(class_names))]

plt.figure(figsize=(6, 4))
bars = plt.bar(class_names, counts, color="steelblue")
plt.bar_label(bars)
plt.title("Class distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()

## 8.1 Look at random images

Numbers are not enough. Actually **look** at a random sample — this is how you catch mislabelled
files, duplicates, and images that are not what you expected.

In [ ]:
random_indices = random.sample(range(len(image_paths)), k=min(6, len(image_paths)))

sample_images = []
sample_titles = []
for index in random_indices:
    image = cv2.imread(image_paths[index])
    sample_images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    sample_titles.append(f"{class_names[labels[index]]}\n{os.path.basename(image_paths[index])}")

show_images(sample_images, sample_titles, columns=3)

## 8.2 Class imbalance

Our dataset is **balanced** — cats and dogs come in equal numbers, and our sampling kept it that way.
That is a comfortable position to be in, and it is also unusual. Most datasets you collect yourself
will not look like this, so it is worth understanding what you are being spared.

A dataset is **imbalanced** when some classes have far more examples than others:

```text
Class A → 900 images

Class B → 100 images
```

> **What could happen if we train a model on highly imbalanced data?**

Think about it before reading on.

A model that always predicts *Class A* would be **90% accurate** on that dataset — while being
completely useless. It never learns anything about Class B, because ignoring Class B barely costs it
anything during training. Worse, plain accuracy will *hide* the problem from you: 90% looks like success.

This matters most when the rare class is the one you care about — fraud, disease, defects.

There are ways to deal with this (resampling, class weights, better metrics like precision/recall
and the confusion matrix). We are **not** implementing them today. For now the goal is **awareness**:

> Always check your class distribution before you trust an accuracy number.

---

# 9. Train / Validation / Test Split

```text
        Full Dataset
             ↓
      ┌──────┼──────┐
      ↓      ↓      ↓
    Train   Val    Test
     70%    15%    15%
```

| Split | Purpose | Model learns from it? |
|---|---|---|
| **Train** | the model updates its weights on these images | yes |
| **Validation** | check performance during training, tune settings, decide when to stop | no — but *we* look at it repeatedly |
| **Test** | the final, honest estimate of performance | no — touched **once**, at the very end |

Why do we need both validation and test? Because every time you look at a score and change
something, you leak a little information about that data into your decisions. The validation set
gets "used up" that way. The test set stays sealed so the final number means something.

We fix the random seed so the split is **reproducible**: the same images always land in the same split.

In [ ]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15   # whatever is left over


def split_dataset(image_paths, labels, train_ratio, val_ratio, seed=SEED):
    """Shuffle the examples once and cut them into three groups."""
    indices = np.arange(len(image_paths))

    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    n_total = len(indices)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_indices = indices[:n_train]
    val_indices = indices[n_train:n_train + n_val]
    test_indices = indices[n_train + n_val:]

    def take(selected):
        return ([image_paths[i] for i in selected], [labels[i] for i in selected])

    return take(train_indices), take(val_indices), take(test_indices)


(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels) = split_dataset(
    image_paths, labels, TRAIN_RATIO, VAL_RATIO
)

print("Total images      :", len(image_paths))
print()
print(f"Train images      : {len(train_paths):>4}  ({100 * len(train_paths) / len(image_paths):.1f}%)")
print(f"Validation images : {len(val_paths):>4}  ({100 * len(val_paths) / len(image_paths):.1f}%)")
print(f"Test images       : {len(test_paths):>4}  ({100 * len(test_paths) / len(image_paths):.1f}%)")

### Check the class balance *inside* each split

A shuffle is random, so a split can accidentally end up with very few examples of a rare class.
Always verify.

In [ ]:
print(f"{'split':<12}" + "".join(f"{name:>12}" for name in class_names))
for split_name, split_labels in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    row = [int(np.sum(np.array(split_labels) == index)) for index in range(len(class_names))]
    print(f"{split_name:<12}" + "".join(f"{count:>12}" for count in row))

**Challenge:** change `TRAIN_RATIO` to `0.6` and re-run. How many images does each split have now?
What happens to the smallest class in the test split?

*(If a class ends up with very few examples in a split, the fix is a **stratified** split — splitting
each class separately and then combining. Try implementing it if you want a harder exercise.)*

---

# 10. Understanding Data Leakage

**Data leakage** is when information from the test set sneaks into training. The result is a model
that looks brilliant in your notebook and fails in the real world — the worst possible failure mode,
because you do not find out until it is deployed.

The simplest form of leakage is the same image appearing in two splits.

Let's build a *broken* split on purpose so you can see it.

In [ ]:
# WRONG on purpose: each split is drawn from the full dataset independently.
rng = np.random.default_rng(SEED)

leaky_train = [image_paths[i] for i in rng.choice(len(image_paths), size=int(0.7 * len(image_paths)), replace=False)]
leaky_test = [image_paths[i] for i in rng.choice(len(image_paths), size=int(0.15 * len(image_paths)), replace=False)]

overlap = set(leaky_train) & set(leaky_test)

print("Leaky train size :", len(leaky_train))
print("Leaky test size  :", len(leaky_test))
print("Images in BOTH   :", len(overlap))
print()
for path in list(overlap)[:3]:
    print("  leaked:", path)

Those overlapping images are ones the model **memorised during training** and is now being graded on.
The test score is measuring memory, not learning.

The rule we want is simply:

```text
Train Images
     ∩
Test Images
     =
Empty
```

Our real split shuffles once and cuts, so no index can land in two places. Let's prove it with Python sets
instead of trusting the code.

In [ ]:
train_set = set(train_paths)
val_set = set(val_paths)
test_set = set(test_paths)

print("Images in both train and val  :", len(train_set & val_set))
print("Images in both train and test :", len(train_set & test_set))
print("Images in both val   and test :", len(val_set & test_set))
print()

no_overlap = not (train_set & val_set or train_set & test_set or val_set & test_set)
all_accounted_for = len(train_set | val_set | test_set) == len(set(image_paths))

print("No overlap between splits :", no_overlap)
print("Every image used exactly once:", all_accounted_for)

> **The test set should contain data the model has not seen during training.**

### Leakage is sneakier than duplicate file names

Identical paths are the easy case. In real projects leakage usually hides:

- **Near-duplicates** — 30 frames extracted from the same video clip, or the same product photographed
  twice. Different file names, essentially the same image. Split by *video* or *product*, not by *frame*.
  Our cats-and-dogs images were scraped from the web, so several near-identical shots of the **same pet**
  almost certainly exist under different file names. Our split does not catch that — worth knowing.
- **The same person / patient / location** in both train and test, when the model can recognise them.
- **Augmenting before splitting** — you generate flipped copies, save them, *then* split. The original
  lands in train and its flip lands in test. This is why we augment **inside** the pipeline, after
  the split, and never write augmented files to disk.
- **Computing normalization statistics on the whole dataset** — the mean and std then carry a little
  information about your test images.

**Challenge:** write a check that no two splits contain files with the same **base name**
(`os.path.basename`), even if they live in different folders.

---

# 11. Building a Custom PyTorch Dataset

This is the heart of the notebook.

> **PyTorch needs a consistent way to access one training example at a time.**

PyTorch does not care where your data comes from — files, a database, the internet. It only asks you
to answer two questions:

1. *How many examples are there?*
2. *Given an index `i`, give me example number `i`.*

If your class can answer those two questions, every PyTorch tool works with it. That contract is
expressed as three methods:

```python
class CustomImageDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):
        pass

    def __len__(self):
        pass

    def __getitem__(self, index):
        pass
```

> `torchvision` does ship a ready-made `ImageFolder` class that does all of this for you.
> We are deliberately **not** using it, because when something goes wrong in your data you need to
> know what is happening inside.

## 11.1 `__init__` — store what we need

This method runs **once**, when the dataset is created. It should be cheap: it stores references,
it does not load images.

It stores:

- **image paths** — the list of file locations
- **labels** — the integer label for each path (same order!)
- **transform** — the preprocessing/augmentation pipeline to apply to each image

Loading all images here would blow up your memory on any realistic dataset. We load lazily, one image
at a time, in `__getitem__`.

## 11.2 `__len__` — how big is the dataset

> **This tells PyTorch how many examples exist in the dataset.**

The `DataLoader` uses it to know how many batches to create and which indices are valid.

```python
def __len__(self):
    return len(self.image_paths)
```

## 11.3 `__getitem__` — return one example

This is where the real work happens. Given an index, produce **one (image, label) pair**:

```text
Index
 ↓
Get Image Path
 ↓
Load Image
 ↓
Convert BGR → RGB
 ↓
Apply Transform
 ↓
Get Label
 ↓
Return Image + Label
```

Every single step matters:

- **Load** with `cv2.imread` — returns a NumPy array, or `None` if the file is broken
- **Convert BGR → RGB** — OpenCV's channel order is not the one everything else expects
- **Apply transform** — resize, augment, convert to tensor, normalize
- **Return** `image, label` — a tuple. This is the fundamental unit PyTorch will use.

In [ ]:
class CustomImageDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        label = self.labels[index]

        return image, label

That is the whole thing — about fifteen lines. Everything else in PyTorch's data machinery is built
on top of these three methods.

The `if image is None` guard should never fire, because we already dropped the unreadable files back
in section 3.6. It stays anyway: a check that costs nothing and turns a baffling crash deep inside
the `DataLoader` into a message that names the offending file is always worth keeping.

Note that `__getitem__` returns the label as a plain Python `int`. The `DataLoader` will collect the
labels of a whole batch and turn them into a tensor automatically.

---

# 12. Preprocessing vs Augmentation

We need **two different pipelines**, and the difference is one of the most common beginner mistakes:

```text
TRAIN                      VALIDATION / TEST

Resize                     Resize
↓                          ↓
Augmentation               Tensor
↓                          ↓
Tensor                     Normalization
↓
Normalization
```

**Training** gets augmentation, so the model sees varied data and generalises better.

**Validation and test** get **no random augmentation**. They must be deterministic: if the images
changed randomly every time, your validation score would jump around for no reason and you could
never tell whether a change actually improved the model.

What both pipelines share — resize, tensor conversion, normalization — is **preprocessing**: it must
be identical, or the model is being fed differently at evaluation time than at training time.

In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("TRAIN transform")
print(train_transform)
print()
print("VALIDATION / TEST transform")
print(eval_transform)

Order matters inside a `Compose`:

- `ToPILImage()` comes **first** because our images arrive as NumPy arrays from OpenCV
- the random augmentations work on the PIL image
- `ToTensor()` converts to `C × H × W` floats in `[0, 1]`
- `Normalize()` must come **after** `ToTensor()`, because it operates on a tensor

Swapping `ToTensor()` and `Normalize()` is a guaranteed error — a useful one to make once, on purpose.

---

# 13. Create the Datasets

Now we combine everything: the split paths, the labels, and the right transform for each split.

In [ ]:
train_dataset = CustomImageDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CustomImageDataset(val_paths, val_labels, transform=eval_transform)
test_dataset = CustomImageDataset(test_paths, test_labels, transform=eval_transform)

print("Train dataset size      :", len(train_dataset))
print("Validation dataset size :", len(val_dataset))
print("Test dataset size       :", len(test_dataset))

`len(train_dataset)` works because we implemented `__len__`. Now let's ask for a single example —
this calls our `__getitem__`.

In [ ]:
image, label = train_dataset[0]

print("Image type  :", type(image))
print("Image dtype :", image.dtype)
print("Image shape :", tuple(image.shape))
print("Value range : [{:.2f}, {:.2f}]".format(image.min(), image.max()))
print()
print("Label       :", label, "->", class_names[label])

## The shape is the whole point

```text
[3, 224, 224]

 3   → channels (R, G, B)
 224 → height
 224 → width
```

After `ToTensor()`, the image follows the **Channels × Height × Width** convention.

Compare that with Session 1, where the same image was:

```text
(224, 224, 3)   ->  Height × Width × Channels
```

Same numbers, different order. NumPy/OpenCV put channels last; PyTorch puts channels first.
This is the single most common source of confusing shape errors in computer vision code, and this
line right here is the bridge between Session 1's "images are arrays" and PyTorch's tensors.

Also notice the value range is roughly `[-1, 1]`, not `[0, 255]` — that is our `Normalize` step.

### Augmentation happens on every access

Because the transform is applied inside `__getitem__`, asking for the *same* index twice gives two
*different* tensors in the training set — but identical tensors in the validation set.

In [ ]:
train_a, _ = train_dataset[0]
train_b, _ = train_dataset[0]

val_a, _ = val_dataset[0]
val_b, _ = val_dataset[0]

print("Train: same index twice -> identical tensors?", torch.equal(train_a, train_b))
print("Val  : same index twice -> identical tensors?", torch.equal(val_a, val_b))

---

# 14. Understanding the DataLoader

> **Loading one image at a time is inefficient during model training.**

Models process many images at once — it is faster on a GPU, and the averaged update is more stable.
The `DataLoader` wraps a `Dataset` and takes care of grouping, shuffling and stacking.

```text
Dataset
   ↓
DataLoader
   ↓
Batch
   ↓
Model
```

Two arguments matter most:

- **`batch_size`** — how many examples per batch. Bigger means faster and smoother, but uses more
  memory. `32` is a common default.
- **`shuffle`** — reorder the data every epoch. **`True` for training**, so the model never sees the
  examples in the same order (and never gets a batch that is all one class). **`False` for validation
  and test**, where order is irrelevant and reproducibility is nicer.

`num_workers=0` means images are loaded in the main process. On Windows, values above 0 require the
notebook code to be guarded, so we keep it simple here.

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"Train      : {len(train_dataset):>4} images -> {len(train_loader):>3} batches")
print(f"Validation : {len(val_dataset):>4} images -> {len(val_loader):>3} batches")
print(f"Test       : {len(test_dataset):>4} images -> {len(test_loader):>3} batches")

The number of batches is `ceil(number of images / batch_size)` — the last batch is usually smaller
than the others, because the dataset rarely divides evenly.

---

# 15. Inspect a Batch

`iter(train_loader)` starts iterating and `next(...)` pulls out the first batch.
(During training you would normally write `for images, labels in train_loader:` — this is just a peek.)

In [ ]:
images, labels_batch = next(iter(train_loader))

print("Images:", tuple(images.shape))
print("Labels:", tuple(labels_batch.shape))
print()
print("Images dtype:", images.dtype)
print("Labels dtype:", labels_batch.dtype)
print()
print("Labels in this batch:", labels_batch.tolist())

Breaking the image shape down:

```text
Images: [32, 3, 224, 224]

32  → number of images in the batch
3   → channels
224 → height
224 → width
```

```text
Labels: [32]

one integer label per image
```

The `DataLoader` did two things for us here: it called `__getitem__` 32 times, and it **stacked** the
results into a single tensor with a new first dimension. It also converted our plain Python `int`
labels into a `torch.int64` tensor.

That 4-dimensional tensor `[batch, channels, height, width]` is exactly what a CNN expects as input.

## Visualizing a normalized batch

We cannot display these tensors directly — two things are in the way:

1. The shape is `C × H × W`, but Matplotlib wants `H × W × C` → use `.permute(1, 2, 0)`
2. The values are roughly `[-1, 1]` because of `Normalize` → **undo** the normalization

Undoing normalization is just reversing the formula:

```text
normalized = (value - mean) / std
value      = normalized * std + mean
```

In [ ]:
MEAN = np.array([0.5, 0.5, 0.5])
STD = np.array([0.5, 0.5, 0.5])


def denormalize(tensor):
    """Turn a normalized C x H x W tensor back into a displayable H x W x C image."""
    image = tensor.permute(1, 2, 0).numpy()   # C, H, W  ->  H, W, C
    image = image * STD + MEAN                # undo the normalization
    return np.clip(image, 0, 1)               # remove tiny rounding errors outside [0, 1]


batch_images = [denormalize(images[i]) for i in range(6)]
batch_titles = [class_names[labels_batch[i]] for i in range(6)]

show_images(batch_images, batch_titles, columns=3)

These are real training images as the model would receive them: resized, augmented, converted to
tensors and normalized. If you re-run the cell above, you get a different batch (because
`shuffle=True`) with different augmentations.

**Challenge:** remove the `np.clip` and see what Matplotlib complains about. Why is clipping necessary?

---

# 16. Our Image Data Pipeline

Everything we built, in one picture:

```text
Raw Image
    ↓
Load with OpenCV
    ↓
Convert BGR → RGB
    ↓
Resize
    ↓
Augmentation (Train Only)
    ↓
Convert to Tensor
    ↓
Normalization
    ↓
Custom PyTorch Dataset
    ↓
DataLoader
    ↓
Batch of Images
```

And where each piece lives in our code:

| Pipeline stage | Our code |
|---|---|
| Get the data | `kagglehub.dataset_download(...)` + `find_class_root(...)` |
| Find the files, skip empty ones | `collect_image_paths(...)` |
| Throw out anything that will not decode | `keep_readable_images(...)` |
| Split honestly | `split_dataset(...)` + the set-intersection check |
| Load and convert colour | inside `CustomImageDataset.__getitem__` |
| Resize / augment / tensor / normalize | `train_transform`, `eval_transform` |
| One example at a time | `CustomImageDataset` |
| Many examples at a time | `DataLoader` |

> **We have not trained a model yet.**

But the data is ready. `images` is a `[32, 3, 224, 224]` tensor and `labels` is a `[32]` tensor —
and that is precisely the input a CNN takes.

In [ ]:
print("PIPELINE CHECK")
print("-" * 40)
print(f"Classes            : {class_names}")
print(f"Usable images      : {len(image_paths)}")
print(f"Broken files found : {len(empty_files) + len(broken_paths)}")
print(f"Train / Val / Test : {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}")
print(f"No split overlap   : {no_overlap}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Batch tensor shape : {tuple(images.shape)}")
print(f"Label tensor shape : {tuple(labels_batch.shape)}")
print("-" * 40)
print("Ready for a model.")

---

# 17. Student Challenges

Work through these in new cells below. Every one of them is a small change to something we already
built — that is the point of having a reusable pipeline.

## Challenge 1 — Change the Image Size

Change `IMAGE_SIZE` from `224` to something else (`64`, `128`, `299`), rebuild the transforms and the
datasets, and pull a fresh batch.

Inspect how the tensor shape changes:

```python
image, label = train_dataset[0]
print(image.shape)
```

Questions to answer:

- Which numbers in `[3, 224, 224]` changed, and which stayed the same? Why?
- How does the batch tensor shape change?
- Which size loads faster? Which keeps more detail?

## Challenge 2 — Add an Augmentation

Add **one** more transformation to `train_transform`. Some options:

```python
transforms.RandomVerticalFlip()
transforms.ColorJitter(saturation=0.3, hue=0.1)
transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0))
transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))
```

Then visualize several augmented versions of the same image (reuse the code from section 7).

> **Does this augmentation make sense for your dataset?**

Justify your answer in one sentence. Remember: the augmented image must still deserve the same label.

## Challenge 3 — Inspect Class Distribution

Print the number of images for every class, then decide: **is your dataset balanced?**

A rough guide:

- ratio below ~2:1 → fine
- ratio around 5:1 → be careful, and do not trust plain accuracy
- ratio above 10:1 → you need a strategy

Also print the distribution **per split**. Does the test set represent the full dataset fairly?

**Then break it on purpose.** Cats and dogs arrive balanced, so create an imbalance yourself: keep
all the images of one class but only 10% of the other, re-run the split, and look at the per-split
counts.

```python
cat_paths = [p for p, l in zip(image_paths, labels) if l == 0]
dog_paths = [p for p, l in zip(image_paths, labels) if l == 1]
# now keep only a slice of one of them and rebuild image_paths / labels
```

If a model always guessed the majority class, what accuracy would it get on your new dataset?
Compute that number — it is the score your real model has to beat before it has learned anything at all.

## Challenge 4 — Change the Batch Size

Try:

```text
16
32
64
```

Each time, recreate `train_loader` and inspect:

```python
images, labels = next(iter(train_loader))
print(images.shape)
print(len(train_loader))
```

Which number changes? Which stays the same? What happens to the *number of batches*?
And what does the very last batch look like when the dataset does not divide evenly?

## Challenge 5 — Build Your Own Pipeline

Take a dataset **you** care about and fill in this diagram:

```text
Raw Image
 ↓
?
 ↓
?
 ↓
Tensor
 ↓
Dataset
 ↓
DataLoader
```

Decide, and write down your reasoning:

- **What resizing is needed?** What is the natural size of your images? Are they all the same shape?
- **What augmentation makes sense?** And which augmentation would be actively harmful?
- **What data leakage risks exist?** Are there near-duplicates, multiple photos of the same subject,
  frames from the same video?
- **Is the dataset balanced?** If not, which class is rare, and is it the class you care about?

Then run this notebook against it: skip the kagglehub cells and set `DATA_DIR` to your own
folder-per-class directory, as described in section 3.4. Everything downstream should work unchanged —
that is what "reusable pipeline" actually means.

---

# 18. What Did We Build?

In this notebook you built:

- a **dataset loader** — downloaded 25,000 real photographs with `kagglehub`, found the class folders,
  sampled them reproducibly, and threw out the corrupted files
- an **image preprocessing pipeline** — resize, scale, normalize
- an **augmentation pipeline** — flips, rotations, colour jitter, applied on the fly to training data only
- **train / validation / test splits** — reproducible, and verified free of overlap
- a **custom PyTorch `Dataset`** — `__init__`, `__len__`, `__getitem__`, written from scratch
- **DataLoaders** — batching and shuffling
- **batches of images** — `[32, 3, 224, 224]` tensors, ready for a model

Along the way you also explored the classical toolbox: thresholding, blurring, edge detection,
contours and morphological operations.

The ideas worth carrying forward:

| Idea | Why it matters |
|---|---|
| Real data is broken | corrupted files exist; validate before you train, not during |
| Images are arrays; models want tensors | `H × W × C` → `C × H × W` |
| Consistency beats cleverness | every image the same size, the same value range |
| Augment training data only | validation must be deterministic |
| Splits must not overlap | otherwise your score is a lie |
| Look at your data first | class imbalance and bad labels hide in plain sight |

> **Before training a Computer Vision model, we need to make sure the data is prepared correctly.**

---

```text
Images
 ↓
Data Pipeline
 ↓
Batches
 ↓
 ?
```

We have everything up to the last arrow.

> **What happens inside the model?**